# Get example datasets from CELLxGENE (non immune datasets)

In [ ]:
import os

import anndata
import scanpy as sc
import pandas as pd
import numpy as np
import tqdm

from os.path import join

## Datasets to download

In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class CxGCollection:
    collection_id: str
    dataset_ids: List[str]
    celltype_cols: List[str]
    cell_type_author: str  # this is the most fine-grained annotation provided by the author
    sample_id: str  # this sequencing sample ID

In [ ]:
cxg_collections = [
    CxGCollection(
        collection_id="0f528c8a-a25c-4840-8fa3-d156fa11086f",
        dataset_ids=["1c360b0b-eb2f-45a3-aba9-056026b39fa5"],
        celltype_cols=["author_cell_type", "clusterClass"],
        cell_type_author="author_cell_type",
        sample_id="SampleID"
    ),
    CxGCollection(
        collection_id="2d40e6a7-f2fd-49ba-9db9-6b97e4c6dad5_Immune",
        dataset_ids=["bb9991a6-2532-4bad-b36f-23824bdcdc1b"],
        celltype_cols=["annotation_level1", "annotation_level2"],
        cell_type_author="annotation_level2",
        sample_id="SampleID"
    ),
    CxGCollection(
        collection_id="2d40e6a7-f2fd-49ba-9db9-6b97e4c6dad5",
        dataset_ids=["fddcbc62-68e7-40ff-ab1e-3d99d4145737"],
        celltype_cols=["annotation_level0", "annotation_level1", "annotation_level2"],
        cell_type_author="annotation_level2",
        sample_id="SampleID"
    ),
    CxGCollection(
        collection_id="48259aa8-f168-4bf5-b797-af8e88da6637_Immune",
        dataset_ids=["872b9ff1-86e9-4a73-8499-5aa9a1f3ee4a"],
        celltype_cols=["level0", "level1", "level2"],
        cell_type_author="level2",
        sample_id="SampleID"
    ),
    CxGCollection(
        collection_id="0c8a364b-97b5-4cc8-a593-23c38c6f0ac5",
        dataset_ids=[
            "305fcc76-0c7f-4a5e-949f-12fb2cb7a05d",
            "670db0f3-48d9-4e8b-ada3-8cb6443982cf",
            "016d0c65-abbe-4048-b1ba-6bf7a7c34eae"
        ],
        celltype_cols=["author_cell_type"],
        cell_type_author="author_cell_type",
        sample_id="sample_uuid"
    ),
    CxGCollection(
        collection_id="71f4bccf-53d4-4c12-9e80-e73bfb89e398",
        dataset_ids=["221dff56-a47d-4563-90ed-51b60e2f16d5"],
        celltype_cols=["Annotated Cell Sets"],
        cell_type_author="Annotated Cell Sets",
        sample_id="Sample ID"
    ),
    CxGCollection(
        collection_id="f6c50495-3361-40ed-a819-fb9644396ed9",
        dataset_ids=["8674c375-ae3a-433c-97de-3c56cf8f7304"],
        celltype_cols=["AuthorCellType"],
        cell_type_author="AuthorCellType",
        sample_id="donor_id"
    ),
    CxGCollection(
        collection_id="5c868b6f-62c5-4532-9d7f-a346ad4b50a7",
        dataset_ids=[
            "485d1aee-db62-4373-854c-12a34237e97b",
            "ef7d48a5-c56b-4f13-9903-fb3327924445"
        ],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        sample_id="biosample_id"
    ),
]

## Download raw datasets

In [ ]:
DOWNLOAD_PATH = "/mnt/dssfs02/dataset-similarity/raw"

In [ ]:
for collection in tqdm.tqdm(cxg_collections):
    for dataset in collection.dataset_ids:
        save_path = join(DOWNLOAD_PATH, f"{dataset}.h5ad")
        if not os.path.isfile(save_path):
            os.system(
                f"wget -q -P {DOWNLOAD_PATH} https://datasets.cellxgene.cziscience.com/{dataset}.h5ad"
            )
        else:
            print(f"File {dataset} already exists. Skipping...")


## Preprocess datasets

In [ ]:
import h5py
import numpy as np
from anndata.experimental import read_elem
from pandas.api.types import is_string_dtype
from scipy.sparse import csc_matrix, csr_matrix

In [ ]:
def streamline_count_matrix(x_raw, gene_names_raw, gene_names_ref):
    assert len(gene_names_raw) == len(set(gene_names_raw))
    assert len(gene_names_ref) == len(set(gene_names_ref))
    assert len(gene_names_raw) == x_raw.shape[1]
    assert np.isin(gene_names_raw, gene_names_ref).sum() == x_raw.shape[1]
    # For fast column-wise slicing matrix has to be in csc format
    assert isinstance(x_raw, csc_matrix)
    gene_names_raw, gene_names_ref = np.array(gene_names_raw), np.array(gene_names_ref)
    row, col = np.empty(x_raw.nnz, dtype='i8'), np.empty(x_raw.nnz, dtype='i8')
    data = np.empty(x_raw.nnz, dtype='f4')

    ctr = 0    
    for i, gene in enumerate(gene_names_ref):
        if gene in gene_names_raw:
            gene_idx = np.where(gene == gene_names_raw)[0]
            assert gene_idx.size == 1
            gene_idx = gene_idx[0]
            x_col = x_raw[:, gene_idx]
            idxs_nnz = x_col.indices
            n_nnz = len(idxs_nnz)
            col[ctr:ctr+n_nnz] = i
            row[ctr:ctr+n_nnz] = idxs_nnz
            data[ctr:ctr+n_nnz] = x_col.data
            ctr += n_nnz

    return csr_matrix(
        (data, (row, col)),
        shape=(x_raw.shape[0], len(gene_names_ref)),
        dtype='f4'
    )


In [ ]:
datasets = []
for collection in cxg_collections:
    for dataset in collection.dataset_ids:
        datasets.append(join(DOWNLOAD_PATH, f"{dataset}.h5ad"))

var_dfs = []
for dataset in datasets:
    with h5py.File(dataset) as f:
        var = read_elem(f["var"])[["feature_name"]]
        var.index.name = "feature_id"
        var_dfs.append(var)

var_concat = (
    pd.concat(var_dfs)
    .reset_index()
    .drop_duplicates()
    .set_index("feature_id")
    .sort_index()
)
var_concat

In [ ]:
# columns to keep for the preprocessed data
COLUMNS = [
    "assay", "cell_type", "development_stage", "disease", "donor_id", 
    "is_primary_data", "sex", "suspension_type", "tissue",
]


def preprocess_dataset(
    dataset_path: str, 
    var_set: pd.DataFrame, 
    columns: List[str], 
    cell_type_column: str,
    sample_id_column: str
):
    with h5py.File(dataset_path) as f:
        var = read_elem(f["var"])
        obs = read_elem(f["obs"])
        try:
            x = read_elem(f["raw"]["X"]).astype("f4").tocsc()
        except KeyError:
            # if raw doesn't exist -> use .X instead
            # according to CELLxGENE schema
            # https://github.com/chanzuckerberg/single-cell-curation/blob/main/schema/3.0.0/schema.md#x-matrix-layers
            x = read_elem(f["X"]).astype("f4").tocsc()

    # align feature spaces across datasets
    x = streamline_count_matrix(x, var.index.tolist(), var_set.index.tolist())
    # subselect to desired obs columns
    obs = (
        obs[columns].copy()
        .assign(cell_type_author=lambda df: df[cell_type_column])
        .assign(sample_id=lambda df: df[sample_id_column])
    )
    # convert all columns with string dtype to categorical dtype
    for col in obs.columns:
        if is_string_dtype(obs[col]):
            obs[col] = obs[col].astype("category")

    return anndata.AnnData(X=x, obs=obs, var=var_set)


In [ ]:
SAVE_PATH = "/mnt/dssfs02/dataset-similarity/preprocessed"


for collection in tqdm.tqdm(cxg_collections):
    save_path = join(SAVE_PATH, f"{collection.collection_id}.h5ad")
    if not os.path.isfile(save_path):
        adatas = []
        for dataset in collection.dataset_ids:
            adata = preprocess_dataset(
                join(DOWNLOAD_PATH, f"{dataset}.h5ad"),
                var_concat,
                COLUMNS + collection.celltype_cols,
                collection.cell_type_author,
                collection.sample_id
            )
            adatas.append(adata)

        if len(adatas) > 1:
            adatas = anndata.concat(adatas)
            adatas.var = var_concat
        else:
            adatas = adatas[0]

        adatas.write(save_path, compression="gzip")
